# 關聯人揭露：資金貸與與背書保證

從 MOPS 追蹤關聯人交易：資金貸與、背書保證與擔保。

In [4]:
from twmops import DisclosureFetcher
import pandas as pd

## 取得揭露資料

取得揭露資料後，可用多種方式分析。

In [5]:
fetcher = DisclosureFetcher()

try:
    # Fetch TSMC (2330) disclosure data
    result = fetcher.get_disclosure("2330", year=115, month=3)
except Exception as e:
    print(f"Error fetching disclosure data: {e}")
    print("Note: This may be a temporary MOPS server issue. Please try again later.")
    result = None

if result:
    print(f"{result.company_name} ({result.stock_id})")
    print(f"Report date: {result.year}/{result.month}")
    print()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


台積電 (2330)
Report date: 115/3



## 資金貸與紀錄

## 欄位參考

**資金貸與 (FundsLending)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `current_month`: 本月貸與金額（新台幣千元）
- `previous_month`: 上月貸與金額（新台幣千元）
- `max_limit`: 最高額度（新台幣千元）

**背書保證 (EndorsementGuarantee)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `monthly_change`: 本月變化（新台幣千元）
- `accumulated_balance`: 累計餘額（新台幣千元）
- `max_limit`: 最高額度（新台幣千元）

**跨公司擔保 (CrossCompanyGuarantee)：**
- `parent_to_subsidiary`: 母公司對子公司擔保（新台幣千元）
- `subsidiary_to_parent`: 子公司對母公司擔保（新台幣千元）

**中國擔保 (ChinaGuarantee)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `monthly_change`: 本月變化（新台幣千元）
- `accumulated_balance`: 累計餘額（新台幣千元）

In [6]:
if result:
    print(f"{result.company_name} 關聯人交易摘要")
    print(f"報告日期: {result.year}/{result.month}")
    print()

    # 資金貸與摘要
    if result.funds_lending:
        total_current = sum(l.current_month or 0 for l in result.funds_lending)
        print(f"資金貸與:")
        print(f"  總紀錄數: {len(result.funds_lending)}")
        print(f"  本月合計: NT${total_current:,} 千元")
        print()

    # 背書保證摘要
    if result.endorsement_guarantee:
        total_balance = sum(g.accumulated_balance or 0 for g in result.endorsement_guarantee)
        print(f"背書保證:")
        print(f"  總紀錄數: {len(result.endorsement_guarantee)}")
        print(f"  累計餘額合計: NT${total_balance:,} 千元")
        print()

    # 跨公司擔保
    if result.cross_company:
        print(f"跨公司擔保:")
        if result.cross_company.parent_to_subsidiary is not None:
            print(f"  母公司對子公司: NT${result.cross_company.parent_to_subsidiary:,} 千元")
        if result.cross_company.subsidiary_to_parent is not None:
            print(f"  子公司對母公司: NT${result.cross_company.subsidiary_to_parent:,} 千元")
        print()

    # 中國營運
    if result.china_guarantee:
        print(f"中國營運擔保:")
        print(f"  總紀錄數: {len(result.china_guarantee)}")
        total_china = sum(c.accumulated_balance or 0 for c in result.china_guarantee)
        print(f"  累計餘額合計: NT${total_china:,} 千元")
        for txn in result.china_guarantee[:3]:
            print(f"    {txn.entity}: NT${txn.accumulated_balance or 0:,} 千元")

Related-Party Transaction Summary for 台積電
Report date: 115/3

Funds Lending:
  Total records: 2
  Total current month: NT$14,991,840 thousand

Endorsement/Guarantee:
  Total records: 2
  Total accumulated balance: NT$691,753,511 thousand

Cross-Company Guarantees:
  Parent to subsidiary: NT$691,753,511 thousand
  Subsidiary to parent: NT$0 thousand

China Operations Guarantee:
  Total records: 2
  Total accumulated balance: NT$0 thousand
    本公司: NT$0 thousand
    子公司: NT$0 thousand


## 關聯人交易摘要

In [7]:
if result and result.endorsement_guarantee:
    print(f"背書保證紀錄: {len(result.endorsement_guarantee)}")
    print()
    for guarantee in result.endorsement_guarantee[:5]:
        print(f"實體: {guarantee.entity}")
        print(f"  有未清餘額: {guarantee.has_balance}")
        if guarantee.monthly_change is not None:
            print(f"  本月變化: NT${guarantee.monthly_change:,} 千元")
        if guarantee.accumulated_balance is not None:
            print(f"  累計餘額: NT${guarantee.accumulated_balance:,} 千元")
        if guarantee.max_limit is not None:
            print(f"  最高額度: NT${guarantee.max_limit:,} 千元")
        print()
    if len(result.endorsement_guarantee) > 5:
        print(f"... 還有 {len(result.endorsement_guarantee) - 5} 筆紀錄")
else:
    print("無背書保證紀錄")

Endorsement/Guarantee Records: 2

Entity: 本公司
  Has balance: True
  Monthly change: NT$17,345,616 thousand
  Accumulated balance: NT$691,753,511 thousand
  Max limit: NT$2,167,838,398 thousand

Entity: 子公司
  Has balance: False
  Monthly change: NT$0 thousand
  Accumulated balance: NT$0 thousand
  Max limit: NT$0 thousand



## 背書保證紀錄

In [8]:
if result and result.funds_lending:
    print(f"資金貸與紀錄: {len(result.funds_lending)}")
    print()
    for lending in result.funds_lending[:5]:
        print(f"實體: {lending.entity}")
        print(f"  有未清餘額: {lending.has_balance}")
        if lending.current_month is not None:
            print(f"  本月: NT${lending.current_month:,} 千元")
        if lending.previous_month is not None:
            print(f"  上月: NT${lending.previous_month:,} 千元")
        if lending.max_limit is not None:
            print(f"  最高額度: NT${lending.max_limit:,} 千元")
        print()
    if len(result.funds_lending) > 5:
        print(f"... 還有 {len(result.funds_lending) - 5} 筆貸與紀錄")
else:
    print("無資金貸與紀錄")

Funds Lending Records: 2

Entity: 本公司
  Has balance: False
  Current month: NT$0 thousand
  Previous month: NT$0 thousand
  Max limit: NT$0 thousand

Entity: 子公司
  Has balance: True
  Current month: NT$14,991,840 thousand
  Previous month: NT$12,835,680 thousand
  Max limit: NT$161,235,966 thousand



## 非同步版本（並行）

在 Jupyter notebook 中使用非同步/等待進行並行請求。

或者，若需要並行請求或處於非同步環境，可使用非同步版本。

## 取得背書保證紀錄

取得背書保證交易。

In [11]:
async def fetch_endorsement_guarantee():
    fetcher = DisclosureFetcher()

    try:
        result = await fetcher.get_disclosure_async("2330", year=115, month=3)
    except Exception as e:
        print(f"取得揭露資料時出錯: {e}")
        print("註: 這可能是暫時的 MOPS 伺服器問題。請稍後重試。")
        return None

    print(f"{result.company_name} 背書保證")
    print(f"總紀錄數: {len(result.endorsement_guarantee)}")

endorsement = await fetch_endorsement_guarantee()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


Endorsement/Guarantee for 台積電
Total records: 2


## 欄位參考

**資金貸與 (FundsLending)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `current_month`: 本月貸與金額（新台幣千元）
- `previous_month`: 上月貸與金額（新台幣千元）
- `max_limit`: 最高額度（新台幣千元）

**背書保證 (EndorsementGuarantee)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `monthly_change`: 本月變化（新台幣千元）
- `accumulated_balance`: 累計餘額（新台幣千元）
- `max_limit`: 最高額度（新台幣千元）

**跨公司擔保 (CrossCompanyGuarantee)：**
- `parent_to_subsidiary`: 母公司對子公司擔保（新台幣千元）
- `subsidiary_to_parent`: 子公司對母公司擔保（新台幣千元）

**中國擔保 (ChinaGuarantee)：**
- `entity`: 實體名稱
- `has_balance`: 是否有未清餘額
- `monthly_change`: 本月變化（新台幣千元）
- `accumulated_balance`: 累計餘額（新台幣千元）